# Sentiment & Sarcasm Classification (en-AU / en-UK)

This notebook implements an end-to-end pipeline for joint **sentiment** and **sarcasm** classification on the
en-AU and en-UK subsets of BESSTIE.

**Contents**

1. Data loading and exploration (`train.csv`, `valid.csv`)
2. Local implementation of the official scoring logic, for evaluating every experiment before submission
3. Classical baseline: TF-IDF + Logistic Regression (CPU-only, no downloads)
4. Cross-validated, per-dialect decision-threshold tuning
5. Word-level data augmentation for the sarcasm minority class
6. Baseline submission file
7. LLM-based paraphrase augmentation for the scarcest slice (en-UK sarcasm)
8. Multi-task Transformer fine-tuning (shared encoder, two classification heads) with leakage-safe cross-validation and checkpointing
9. Per-dialect threshold tuning and blending with the classical model
10. Multi-encoder ensembling
11. Pseudo-labelling utility
12. Error analysis
13. Submission format

**Evaluation metric.** Each task (sentiment, sarcasm) is scored by its *weakest* dialect's macro-F1, and the final score is the
mean of the two task scores. Sarcasm is the harder and more imbalanced task, so most of the modelling effort targets it:
class-imbalance-aware training, per-dialect thresholds, augmentation, and encoder ensembling.

## 0. Setup

Expected folder layout:

```
project_root/
├── data/
│   ├── train.csv                    # labelled training data
│   ├── valid.csv                    # labelled validation set
│   ├── evaluate.py                  # official scoring script
│   └── sample_answer_template.csv   # submission format example
├── notebooks/
│   └── sentiment_sarcasm_pipeline.ipynb   (this file)
└── submissions/                     # generated answer.csv / answer.zip files are written here
```

In [1]:
import os, sys, re, random, zipfile, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Paths (relative to notebooks/ when run from this project layout)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
SUB_DIR = os.path.join(PROJECT_ROOT, "submissions")
os.makedirs(SUB_DIR, exist_ok=True)

# Fallback: if the notebook is moved / run flat, look in the current directory too.
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "data"
    SUB_DIR = "submissions"
    os.makedirs(SUB_DIR, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("SUB_DIR :", SUB_DIR)

# Environment check: determines whether Section 8 (Transformer training) can run.
HAS_TORCH, HAS_TRANSFORMERS, HAS_CUDA = False, False, False
try:
    import torch
    HAS_TORCH = True
    HAS_CUDA = torch.cuda.is_available()
except ImportError:
    pass
try:
    import transformers  # noqa
    HAS_TRANSFORMERS = True
except ImportError:
    pass

print(f"torch installed: {HAS_TORCH} | transformers installed: {HAS_TRANSFORMERS} | CUDA available: {HAS_CUDA}")
if not (HAS_TORCH and HAS_TRANSFORMERS and HAS_CUDA):
    print("\nNote: GPU and/or transformers not fully available in this environment.")
    print("    Section 8 (Transformer training) will print setup instructions instead of training.")
    print("    The classical pipeline (Sections 1-7) runs on CPU.")

DATA_DIR: ../data
SUB_DIR : ../submissions
torch installed: False | transformers installed: False | CUDA available: False

Note: GPU and/or transformers not fully available in this environment.
    Section 8 (Transformer training) will print setup instructions instead of training.
    The classical pipeline (Sections 1-7) runs on CPU.


## 1. Load & explore the data

In [2]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid.csv"))

print("train:", train_df.shape, " valid:", valid_df.shape)
train_df.head(3)


train: (5531, 5)  valid: (757, 5)


,source,variety,text,sentiment,sarcasm
0,google,en-AU,This was one of the best dishes I've EVER had!...,1,0
1,google,en-AU,This Mexican restaurant in Penrith is a great ...,1,0
2,google,en-AU,"This was not to bad, I ordered the big pork ri...",1,0


In [3]:
def class_table(df, name):
    rows = []
    for dialect in ["en-AU", "en-UK"]:
        sub = df[df["variety"] == dialect]
        rows.append({
            "set": name, "variety": dialect, "n": len(sub),
            "sentiment_pos_%": round(100 * sub["sentiment"].mean(), 1),
            "sarcasm_pos_%": round(100 * sub["sarcasm"].mean(), 1),
        })
    return pd.DataFrame(rows)

pd.concat([class_table(train_df, "train"), class_table(valid_df, "valid")], ignore_index=True)


,set,variety,n,sentiment_pos_%,sarcasm_pos_%
0,train,en-AU,2709,46.4,29.7
1,train,en-UK,2822,52.2,7.8
2,valid,en-AU,371,46.6,29.4
3,valid,en-UK,386,52.3,7.8


Two observations shape the modelling decisions below:

- **Sentiment is fairly balanced** in both dialects (roughly 45–55% positive), so it is comparatively easy.
- **Sarcasm is heavily imbalanced, and unevenly so across dialects**: en-UK has far fewer sarcastic examples than en-AU. Because the metric
  takes the weakest dialect per task, a model that ignores en-UK sarcasm can look good on average while scoring poorly on the metric that matters.

## 2. Scoring

`data/evaluate.py` is the official scoring script. The function below reimplements its logic so that any candidate
prediction file can be scored against `valid.csv` directly inside the notebook.

In [4]:
from sklearn.metrics import f1_score

def score_predictions(pred_df: pd.DataFrame, gold_df: pd.DataFrame, verbose: bool = True):
    '''Reimplements data/evaluate.py's logic for in-notebook use.'''
    scores = {}
    dialects = ["en-AU", "en-UK"]
    for task in ["sentiment", "sarcasm"]:
        for dialect in dialects:
            gold_f = gold_df[gold_df["variety"] == dialect][task]
            run_f = pred_df[pred_df["variety"] == dialect][task]
            scores[f"{task}-{dialect}"] = f1_score(gold_f, run_f, average="macro")

    sent_scores = [scores[f"sentiment-{d}"] for d in dialects]
    sarc_scores = [scores[f"sarcasm-{d}"] for d in dialects]
    final_score = (min(sent_scores) + min(sarc_scores)) / 2

    if verbose:
        for k, v in scores.items():
            print(f"  f1-{k}: {v:.4f}")
        print(f"  final score: {final_score:.4f}")
    return scores, final_score

def make_answer_df(source_df: pd.DataFrame, sentiment_pred, sarcasm_pred) -> pd.DataFrame:
    '''Builds an answer.csv-shaped DataFrame matching the required columns/row order.'''
    out = source_df[["source", "variety", "text"]].copy()
    out["sentiment"] = np.asarray(sentiment_pred).astype(int)
    out["sarcasm"] = np.asarray(sarcasm_pred).astype(int)
    return out

def save_submission(answer_df: pd.DataFrame, name: str) -> str:
    '''Writes submissions/<name>.csv and submissions/<name>.zip (with answer.csv inside), returns the zip path.'''
    csv_path = os.path.join(SUB_DIR, f"{name}.csv")
    zip_path = os.path.join(SUB_DIR, f"{name}.zip")
    answer_df.to_csv(csv_path, index=False)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname="answer.csv")
    print(f"Wrote {csv_path}\nWrote {zip_path}  <-- upload this file")
    return zip_path


**Sanity check:** scoring the random-label sample submission (`sample_answer_template.csv`) should give near-chance scores, confirming the scoring function is wired correctly.

In [5]:
sample_pred = pd.read_csv(os.path.join(DATA_DIR, "sample_answer_template.csv"))
print("Sanity check on the random-label sample submission (expect low / near-chance scores):")
_ = score_predictions(sample_pred, valid_df)


Sanity check on the random-label sample submission (expect low / near-chance scores):
  f1-sentiment-en-AU: 0.4956
  f1-sentiment-en-UK: 0.4585
  f1-sarcasm-en-AU: 0.4584
  f1-sarcasm-en-UK: 0.4077
  final score: 0.4331


## 3. Baseline model: TF-IDF + Logistic Regression

A fast, CPU-only baseline with no external downloads. Design choices:

- **Word (1–2 gram) and character (3–5 gram, `char_wb`) TF-IDF**, concatenated. Character n-grams capture orthographic and register cues that word n-grams miss.
- **`class_weight="balanced"`** to counter the sarcasm class imbalance without discarding data.
- **Optional stylistic features**: exclamation and question mark counts, ALL-CAPS word ratio, letter elongation, quote-mark counts, ellipsis counts, text length, and a coarse lexicon polarity score. Each task is evaluated with and without these features, and the better configuration is kept.
- **One independent classifier per task**, since sentiment and sarcasm are related but distinct signals.

In [6]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# A small polarity lexicon (not a trained model). A shallow word-count vote captures the
# surface-level polarity of the wording, without context; it is used below as one
# additional input feature.
_POS_WORDS = set('''good great excellent amazing love loved wonderful fantastic best awesome
perfect nice happy beautiful brilliant superb delicious friendly recommend recommended
outstanding impressive enjoy enjoyed enjoyable pleasant lovely fabulous top charming'''.split())
_NEG_WORDS = set('''bad terrible awful worst hate hated horrible poor disappointing disappointed
rude dirty slow disgusting never again avoid worse unacceptable overpriced broken rip-off
scam waste ridiculous pathetic useless nightmare'''.split())

def naive_polarity(text):
    '''1 = more positive words than negative, 0 = more negative, 0.5 = no lexicon hits.'''
    words = re.findall(r"[a-zA-Z']+", str(text).lower())
    pos = sum(1 for w in words if w in _POS_WORDS)
    neg = sum(1 for w in words if w in _NEG_WORDS)
    if pos == 0 and neg == 0:
        return 0.5
    return 1.0 if pos > neg else 0.0

def extract_style_features(texts):
    '''Cheap, hand-crafted cues correlated with sarcasm/register: punctuation
    intensity, ALL-CAPS ratio, letter elongation, scare-quotes, ellipses, length,
    plus the naive lexicon polarity above (computed from text alone - no label needed,
    so it's safe to use as an input feature at inference time too).'''
    feats = []
    for t in texts:
        t = str(t)
        words = t.split()
        n_words = max(len(words), 1)
        caps_words = sum(1 for w in words if len(w) > 1 and w.isupper())
        feats.append([
            t.count("!"), t.count("?"), caps_words / n_words,
            len(re.findall(r"(.)\1{2,}", t)), t.count('"') + t.count("'"),
            t.count("..."), len(t), n_words, naive_polarity(t),
        ])
    return np.array(feats, dtype=np.float32)

def build_featurizer(fit_texts, use_style=True):
    word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9,
                                sublinear_tf=True, max_features=40000)
    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.9,
                                sublinear_tf=True, max_features=40000)
    word_vec.fit(fit_texts)
    char_vec.fit(fit_texts)
    scaler = StandardScaler().fit(extract_style_features(fit_texts)) if use_style else None

    def transform(texts):
        parts = [word_vec.transform(texts), char_vec.transform(texts)]
        if use_style:
            parts.append(csr_matrix(scaler.transform(extract_style_features(texts))))
        return hstack(parts).tocsr()

    return transform

def per_dialect_f1(task, pred_binary):
    pred_binary = np.asarray(pred_binary)
    res = {}
    for dialect in ["en-AU", "en-UK"]:
        mask = (valid_df["variety"] == dialect).values
        res[dialect] = f1_score(valid_df[task][mask], pred_binary[mask], average="macro")
    res["min"] = min(res["en-AU"], res["en-UK"])
    return res

def fit_predict(task, use_style):
    fn = build_featurizer(train_df["text"], use_style=use_style)
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0, random_state=RANDOM_SEED)
    clf.fit(fn(train_df["text"]), train_df[task])
    return fn, clf.predict_proba(fn(valid_df["text"]))[:, 1]

# Compare word+char TF-IDF alone vs. with stylistic features, per task; keep the better configuration on the validation set.
use_style_choice, valid_proba = {}, {}
for task in ["sentiment", "sarcasm"]:
    best = None
    for use_style in [False, True]:
        fn, proba = fit_predict(task, use_style)
        res = per_dialect_f1(task, proba >= 0.5)
        tag = "word+char+style" if use_style else "word+char only"
        print(f"[{task}] {tag:16s} AU={res['en-AU']:.4f}  UK={res['en-UK']:.4f}  min={res['min']:.4f}")
        if best is None or res["min"] > best[0]:
            best = (res["min"], use_style, proba)
    _, use_style_choice[task], valid_proba[task] = best
    print(f"  -> keeping {'word+char+style' if use_style_choice[task] else 'word+char only'} for {task}\n")

baseline_pred = make_answer_df(
    valid_df,
    sentiment_pred=(valid_proba["sentiment"] >= 0.5).astype(int),
    sarcasm_pred=(valid_proba["sarcasm"] >= 0.5).astype(int),
)
print("TF-IDF + Logistic Regression baseline @ threshold 0.5 (best feature set per task):")
baseline_scores, baseline_final = score_predictions(baseline_pred, valid_df)

[sentiment] word+char only   AU=0.8378  UK=0.9141  min=0.8378


[sentiment] word+char+style  AU=0.8038  UK=0.9091  min=0.8038
  -> keeping word+char only for sentiment



[sarcasm] word+char only   AU=0.7056  UK=0.6208  min=0.6208


[sarcasm] word+char+style  AU=0.7093  UK=0.6444  min=0.6444
  -> keeping word+char+style for sarcasm

TF-IDF + Logistic Regression baseline @ threshold 0.5 (best feature set per task):
  f1-sentiment-en-AU: 0.8378
  f1-sentiment-en-UK: 0.9141
  f1-sarcasm-en-AU: 0.7093
  f1-sarcasm-en-UK: 0.6444
  final score: 0.7411


## 4. Cross-validated, per-dialect threshold tuning

Sarcasm is imbalanced differently in each dialect (en-AU ≈ 30% positive, en-UK ≈ 8%). Since the metric scores each dialect separately and takes the minimum,
a single global threshold is not appropriate. Instead, one threshold is tuned **per dialect** using 5-fold **out-of-fold predictions on `train.csv`**
(the labels in `valid.csv` are never used for tuning), and these thresholds are then applied to `valid.csv`.

In [7]:
from sklearn.model_selection import StratifiedKFold

def tune_thresholds_cv(task: str, use_style: bool, n_splits: int = 5):
    y = train_df[task].values
    variety = train_df["variety"].values
    oof_proba = np.zeros(len(train_df))

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    for tr_idx, va_idx in skf.split(train_df["text"], y):
        fold_featurize = build_featurizer(train_df["text"].iloc[tr_idx], use_style=use_style)
        X_tr = fold_featurize(train_df["text"].iloc[tr_idx])
        X_va = fold_featurize(train_df["text"].iloc[va_idx])
        clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0, random_state=RANDOM_SEED)
        clf.fit(X_tr, y[tr_idx])
        oof_proba[va_idx] = clf.predict_proba(X_va)[:, 1]

    thresholds = {}
    for dialect in ["en-AU", "en-UK"]:
        mask = variety == dialect
        best_thr, best_f1 = 0.5, -1.0
        for thr in np.arange(0.10, 0.91, 0.02):
            pred = (oof_proba[mask] >= thr).astype(int)
            f1 = f1_score(y[mask], pred, average="macro")
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[dialect] = best_thr
        print(f"[{task}] {dialect}: best CV threshold = {best_thr:.2f} (OOF macro-F1 = {best_f1:.4f})")
    return thresholds

def apply_per_dialect_threshold(task, proba, thresholds):
    pred = np.zeros(len(valid_df), dtype=int)
    for dialect, thr in thresholds.items():
        mask = (valid_df["variety"] == dialect).values
        pred[mask] = (np.asarray(proba)[mask] >= thr).astype(int)
    return pred

best_thr = {task: tune_thresholds_cv(task, use_style_choice[task]) for task in ["sentiment", "sarcasm"]}


[sentiment] en-AU: best CV threshold = 0.44 (OOF macro-F1 = 0.8386)
[sentiment] en-UK: best CV threshold = 0.56 (OOF macro-F1 = 0.9139)


[sarcasm] en-AU: best CV threshold = 0.52 (OOF macro-F1 = 0.6836)
[sarcasm] en-UK: best CV threshold = 0.62 (OOF macro-F1 = 0.6193)


In [8]:
tuned_pred = make_answer_df(
    valid_df,
    sentiment_pred=apply_per_dialect_threshold("sentiment", valid_proba["sentiment"], best_thr["sentiment"]),
    sarcasm_pred=apply_per_dialect_threshold("sarcasm", valid_proba["sarcasm"], best_thr["sarcasm"]),
)

print("TF-IDF + Logistic Regression, CV-tuned PER-DIALECT thresholds:")
tuned_scores, tuned_final = score_predictions(tuned_pred, valid_df)

print(f"\n0.5-threshold final score : {baseline_final:.4f}")
print(f"tuned-threshold final score: {tuned_final:.4f}")

best_classical_pred = tuned_pred if tuned_final >= baseline_final else baseline_pred
best_classical_name  = "classical (CV-tuned per-dialect thresholds)" if tuned_final >= baseline_final else "classical (0.5 threshold)"
print(f"\nUsing: {best_classical_name}")

# Record which threshold choice performed better for sentiment, so later sections
# (which only modify the sarcasm model) reuse it consistently.
sentiment_thresholds = best_thr["sentiment"] if tuned_final >= baseline_final else {"en-AU": 0.5, "en-UK": 0.5}

TF-IDF + Logistic Regression, CV-tuned PER-DIALECT thresholds:
  f1-sentiment-en-AU: 0.8288
  f1-sentiment-en-UK: 0.9118
  f1-sarcasm-en-AU: 0.7093
  f1-sarcasm-en-UK: 0.6086
  final score: 0.7187

0.5-threshold final score : 0.7411
tuned-threshold final score: 0.7187

Using: classical (0.5 threshold)


## 5. Word-level data augmentation for the sarcasm minority class

An EDA-style augmentation: for each sarcastic training example, generate noisy variants (random word deletion and random word swap) and add them to the training set.
This oversamples the minority class with some diversity rather than exact duplicates. en-UK sarcasm receives more variants (3 vs. 2) because it is the scarcer dialect.
The effect is evaluated against the non-augmented model before being adopted.

In [9]:
def random_deletion(words, p=0.10):
    if len(words) == 1:
        return words
    kept = [w for w in words if random.random() > p]
    return kept if kept else [random.choice(words)]

def random_swap(words, n=1):
    words = words.copy()
    for _ in range(n):
        if len(words) < 2:
            break
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return words

def augment_text(text, n_aug=2):
    words = text.split()
    variants = []
    for _ in range(n_aug):
        w = random_deletion(words, p=0.10)
        w = random_swap(w, n=max(1, len(w) // 15))
        variants.append(" ".join(w))
    return variants

sarc_pos = train_df[train_df["sarcasm"] == 1]
aug_rows = []
for idx, row in sarc_pos.iterrows():
    n_aug = 3 if row["variety"] == "en-UK" else 2
    for t in augment_text(row["text"], n_aug=n_aug):
        aug_rows.append({"source": row["source"], "variety": row["variety"], "text": t,
                          "sentiment": row["sentiment"], "sarcasm": row["sarcasm"], "_parent_idx": idx})

# _parent_idx records which original train_df row each augmented variant came from (original
# rows point to themselves). Section 8 uses it to keep augmented rows on the training side of
# whichever fold their parent belongs to, so held-out folds are never scored against
# near-duplicates of their own training data.
train_df_with_parent = train_df.copy()
train_df_with_parent["_parent_idx"] = train_df_with_parent.index
train_df_aug = pd.concat([train_df_with_parent, pd.DataFrame(aug_rows)], ignore_index=True)
print(f"train size: {len(train_df)} -> {len(train_df_aug)} after augmenting sarcastic examples")

# Compare on the sarcasm task only (sentiment is balanced and does not need augmentation)
def fit_predict_on(task, data, use_style):
    fn = build_featurizer(data["text"], use_style=use_style)
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0, random_state=RANDOM_SEED)
    clf.fit(fn(data["text"]), data[task])
    return clf.predict_proba(fn(valid_df["text"]))[:, 1]

sarc_proba_aug = fit_predict_on("sarcasm", train_df_aug, use_style_choice["sarcasm"])
res_before = per_dialect_f1("sarcasm", valid_proba["sarcasm"] >= 0.5)
res_after = per_dialect_f1("sarcasm", sarc_proba_aug >= 0.5)
print(f"sarcasm WITHOUT augmentation: AU={res_before['en-AU']:.4f}  UK={res_before['en-UK']:.4f}  min={res_before['min']:.4f}")
print(f"sarcasm WITH augmentation   : AU={res_after['en-AU']:.4f}  UK={res_after['en-UK']:.4f}  min={res_after['min']:.4f}")

if res_after["min"] > res_before["min"]:
    print("\nAugmentation improves the weakest dialect; adopting it for the sarcasm model.")
    valid_proba["sarcasm"] = sarc_proba_aug
    aug_pred = make_answer_df(
        valid_df,
        sentiment_pred=apply_per_dialect_threshold("sentiment", valid_proba["sentiment"], sentiment_thresholds),
        sarcasm_pred=(valid_proba["sarcasm"] >= 0.5).astype(int),
    )
    _, aug_final = score_predictions(aug_pred, valid_df, verbose=False)
    if aug_final > tuned_final and aug_final > baseline_final:
        best_classical_pred, best_classical_name = aug_pred, "classical (+ sarcasm augmentation)"
else:
    print("\nNo improvement; keeping the non-augmented sarcasm model.")

print(f"\nFinal classical choice: {best_classical_name}")

train size: 5531 -> 7798 after augmenting sarcastic examples


sarcasm WITHOUT augmentation: AU=0.7093  UK=0.6444  min=0.6444
sarcasm WITH augmentation   : AU=0.7058  UK=0.6668  min=0.6668

Augmentation improves the weakest dialect; adopting it for the sarcasm model.

Final classical choice: classical (+ sarcasm augmentation)


## 6. Baseline submission

Packages the best classical model's predictions as a valid `answer.zip`.

In [10]:
zip_path = save_submission(best_classical_pred, "dev_classical_baseline")


Wrote ../submissions/dev_classical_baseline.csv
Wrote ../submissions/dev_classical_baseline.zip  <-- upload this file


## 7. LLM paraphrase augmentation for en-UK sarcasm

Word-level augmentation (Section 5) adds limited diversity. LLM-generated paraphrases can vary register and phrasing while preserving the sarcastic intent.
This section generates paraphrases for the scarcest slice, en-UK sarcastic examples in `train.csv`, using the Groq API (OpenAI-compatible chat completions).
It produces new *training* examples only; no test labels are involved.

**Requirements:** `pip install groq` and a `GROQ_API_KEY` environment variable. The default model is `openai/gpt-oss-20b`; a larger model such as `openai/gpt-oss-120b` can be substituted if paraphrase quality needs to improve.

**Caching.** The augmented set is written to `alta2026_checkpoints/train_df_llm_aug.csv` (Google Drive on Colab, otherwise a local `checkpoints/` folder) on first generation
and reloaded on subsequent runs, so API calls are not repeated after a runtime restart. If neither a cache nor an API key is available, `train_df_llm_aug` is set to `None`
and Section 8 falls back to the word-level augmented set from Section 5.

The first cell below installs the SDK and prompts for the API key with hidden input.

In [11]:
%pip install -q groq

import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    try:
        os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY (input hidden, get one at console.groq.com): ")
    except Exception:
        print("Couldn't prompt for input in this environment (e.g. running non-interactively).")
        print('Set it manually instead, then re-run this cell:')
        print('  import os; os.environ["GROQ_API_KEY"] = "gsk_..."')
else:
    print("GROQ_API_KEY already set for this session.")


Couldn't prompt for input in this environment (e.g. running non-interactively).
Set it manually instead, then re-run this cell:
  import os; os.environ["GROQ_API_KEY"] = "gsk_..."


In [12]:
# Persistent cache location for the LLM-augmented training set and model checkpoints.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ALTA_CKPT_ROOT = '/content/drive/MyDrive/alta2026_checkpoints'
except Exception:
    ALTA_CKPT_ROOT = os.path.join(os.path.dirname(SUB_DIR), 'checkpoints')
    print("Google Drive not available; caching locally (this will not persist across a full runtime reset).")
os.makedirs(ALTA_CKPT_ROOT, exist_ok=True)
LLM_AUG_CACHE_PATH = os.path.join(ALTA_CKPT_ROOT, 'train_df_llm_aug.csv')
print(f"Checkpoint root: {ALTA_CKPT_ROOT}")

Google Drive not available; caching locally (this will not persist across a full runtime reset).
Checkpoint root: ../checkpoints


In [13]:
def llm_paraphrase_augment(df, n_variants=2, model="openai/gpt-oss-20b"):
    '''Generates n_variants sarcasm-preserving paraphrases per row via the Groq API
    (OpenAI-compatible chat completions). Returns a DataFrame in the same
    source/variety/text/sentiment/sarcasm/_parent_idx shape as train_df_aug, ready to concat on.'''
    from groq import Groq
    client = Groq()
    rows = []
    for idx, row in df.iterrows():
        prompt = (
            f"Rewrite the following sarcastic text in {n_variants} different way(s), each in a "
            "different register or phrasing, while keeping it just as sarcastic and preserving "
            "its meaning and rough length. Output exactly one rewrite per line, no numbering, "
            "no commentary, nothing else.\n\nText: " + str(row["text"])
        )
        resp = client.chat.completions.create(model=model, max_tokens=300,
                                               messages=[{"role": "user", "content": prompt}])
        variants = [ln.strip() for ln in resp.choices[0].message.content.strip().split("\n") if ln.strip()]
        for v in variants[:n_variants]:
            rows.append({"source": row["source"], "variety": row["variety"], "text": v,
                         "sentiment": row["sentiment"], "sarcasm": row["sarcasm"], "_parent_idx": idx})
    return pd.DataFrame(rows)

HAS_GROQ_PKG = True
try:
    import groq  # noqa
except ImportError:
    HAS_GROQ_PKG = False
HAS_API_KEY = bool(os.environ.get("GROQ_API_KEY"))

if os.path.exists(LLM_AUG_CACHE_PATH):
    train_df_llm_aug = pd.read_csv(LLM_AUG_CACHE_PATH)
    print(f"Loaded cached LLM-augmented training set ({len(train_df_llm_aug)} rows); no API calls needed.")
    print(f"  {LLM_AUG_CACHE_PATH}")
elif HAS_GROQ_PKG and HAS_API_KEY:
    uk_sarcastic = train_df[(train_df["sarcasm"] == 1) & (train_df["variety"] == "en-UK")]
    print(f"Generating LLM paraphrases for {len(uk_sarcastic)} en-UK sarcastic examples...")
    llm_aug_df = llm_paraphrase_augment(uk_sarcastic, n_variants=3)
    train_df_llm_aug = pd.concat([train_df_aug, llm_aug_df], ignore_index=True)
    train_df_llm_aug.to_csv(LLM_AUG_CACHE_PATH, index=False)
    print(f"train size: {len(train_df_aug)} -> {len(train_df_llm_aug)} after LLM paraphrase augmentation")
    print(f"Cached to {LLM_AUG_CACHE_PATH}")
else:
    train_df_llm_aug = None
    print("No cached LLM-augmented data found and the Groq API is not configured.")
    print("Section 8 will fall back to the word-level augmented set from Section 5.")
    print("\nTo generate the LLM-augmented set:")
    print("  1. pip install groq")
    print("  2. Set the GROQ_API_KEY environment variable")
    print(f"  3. Re-run this cell; results are cached to {LLM_AUG_CACHE_PATH}")

if train_df_llm_aug is not None:
    # Compare against the word-level augmentation result using the classical pipeline
    sarc_proba_llm = fit_predict_on("sarcasm", train_df_llm_aug, use_style_choice["sarcasm"])
    res_llm = per_dialect_f1("sarcasm", sarc_proba_llm >= 0.5)
    res_before = per_dialect_f1("sarcasm", valid_proba["sarcasm"] >= 0.5)
    print(f"\nsarcasm WITH LLM augmentation   : AU={res_llm['en-AU']:.4f}  UK={res_llm['en-UK']:.4f}  min={res_llm['min']:.4f}")
    print(f"sarcasm best so far (Section 5)   : AU={res_before['en-AU']:.4f}  UK={res_before['en-UK']:.4f}  min={res_before['min']:.4f}")

No cached LLM-augmented data found and the Groq API is not configured.
Section 8 will fall back to the word-level augmented set from Section 5.

To generate the LLM-augmented set:
  1. pip install groq
  2. Set the GROQ_API_KEY environment variable
  3. Re-run this cell; results are cached to ../checkpoints/train_df_llm_aug.csv


## 8. Multi-task Transformer fine-tuning

A shared Transformer encoder with two linear heads (sentiment and sarcasm), trained with 5-fold cross-validation. Key design points:

- **Training data.** Uses `train_df_llm_aug` (Section 7) if available, otherwise `train_df_aug` (Section 5), otherwise plain `train_df`.
- **Leakage-safe folds.** Folds are assigned using only the *original* `train_df` rows. Each augmented row is added only to the training side of the fold its original parent belongs to, so a held-out fold is never evaluated on near-duplicates of its own training data. Out-of-fold (OOF) arrays remain aligned to the original `train_df` rows.
- **Fold-level and model-level checkpointing.** Each fold saves its held-out predictions and model weights, and a completed model saves a summary file. An interrupted run resumes from the last completed fold, and completed models are reused without retraining.
- **Dataset-tagged checkpoint paths** (`transformer_cv`, `transformer_cv_classical_aug`, `transformer_cv_llm_aug`), so results from different training sets never collide.
- **Mixed precision.** Model weights stay in FP32 and FP16 is used only inside `autocast`, which avoids FP16-gradient unscaling errors.
- **Low-VRAM defaults.** Batch size 4 with gradient accumulation 4 (effective batch size 16), and gradient checkpointing for DeBERTa-family models, so training fits on a single 16 GB GPU such as a T4.

**Requirements:** a GPU and Hugging Face Hub access. The data-preparation cell runs anywhere; the training cell detects missing requirements and prints setup instructions instead of failing.

In [14]:
# ── Build the training pool used by the Transformer, with leakage-safe fold tracking ──
# Priority: LLM-augmented > word-level augmented > plain train_df. Every row carries
# `_parent_idx` pointing back to its original train_df row (original rows point to themselves),
# which keeps fold splitting leakage-safe.
if "train_df_llm_aug" in dir() and train_df_llm_aug is not None:
    TRANSFORMER_TRAIN_DF = train_df_llm_aug.copy()
    TRANSFORMER_DATASET_TAG = "llm_aug"
elif "train_df_aug" in dir():
    TRANSFORMER_TRAIN_DF = train_df_aug.copy()
    TRANSFORMER_DATASET_TAG = "classical_aug"
else:
    TRANSFORMER_TRAIN_DF = train_df.copy()
    TRANSFORMER_TRAIN_DF["_parent_idx"] = TRANSFORMER_TRAIN_DF.index
    TRANSFORMER_DATASET_TAG = "base"

TRANSFORMER_TRAIN_DF = TRANSFORMER_TRAIN_DF.reset_index(drop=True)
N_ORIGINAL_TRAIN = len(train_df)

print(f"Transformer training pool: {len(TRANSFORMER_TRAIN_DF)} rows (tag='{TRANSFORMER_DATASET_TAG}')")
print(f"  {N_ORIGINAL_TRAIN} original + {len(TRANSFORMER_TRAIN_DF) - N_ORIGINAL_TRAIN} augmented")
print(f"OOF scoring uses only the {N_ORIGINAL_TRAIN} original train_df rows; augmentation expands the")
print("training side of each fold, never the held-out side.")

# Checkpoint roots. Reuses ALTA_CKPT_ROOT from Section 7 if already defined; otherwise computes it.
if "ALTA_CKPT_ROOT" not in dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        ALTA_CKPT_ROOT = '/content/drive/MyDrive/alta2026_checkpoints'
    except Exception:
        ALTA_CKPT_ROOT = os.path.join(os.path.dirname(SUB_DIR), 'checkpoints')
    os.makedirs(ALTA_CKPT_ROOT, exist_ok=True)

CKPT_DIR_BY_TAG = {
    "base": os.path.join(ALTA_CKPT_ROOT, "transformer_cv"),
    "classical_aug": os.path.join(ALTA_CKPT_ROOT, "transformer_cv_classical_aug"),
    "llm_aug": os.path.join(ALTA_CKPT_ROOT, "transformer_cv_llm_aug"),
}
for _d in CKPT_DIR_BY_TAG.values():
    os.makedirs(_d, exist_ok=True)
print(f"\nCheckpoint root: {ALTA_CKPT_ROOT}")
print(f"This run's checkpoint dir: {CKPT_DIR_BY_TAG[TRANSFORMER_DATASET_TAG]}")

Transformer training pool: 7798 rows (tag='classical_aug')
  5531 original + 2267 augmented
OOF scoring uses only the 5531 original train_df rows; augmentation expands the
training side of each fold, never the held-out side.

Checkpoint root: ../checkpoints
This run's checkpoint dir: ../checkpoints/transformer_cv_classical_aug


In [15]:
TRANSFORMER_MODEL_NAME = "roberta-base"  # stable, strong default; "microsoft/deberta-v3-base" is a common alternative
TRANSFORMER_BATCH_SIZE = 4
TRANSFORMER_GRAD_ACCUMULATION = 4        # effective batch size 16; sized for a 16 GB GPU
TRANSFORMER_MAX_LEN = 128
TRANSFORMER_LR = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01
TRANSFORMER_WARMUP_RATIO = 0.10
TRANSFORMER_MAX_GRAD_NORM = 1.0
N_FOLDS = 5
EPOCHS = 5

def _safe_ckpt_name(model_name):
    return model_name.replace("/", "__").replace("\\", "__").replace(":", "_").replace(" ", "_")

def _is_deberta(model_name):
    return "deberta" in model_name.lower()

def run_transformer_cv_pipeline(model_name, n_folds=None, epochs=None, dataset_tag=None):
    '''Trains (or resumes / reuses a checkpoint for) one encoder, fold by fold, with both
    fold-level and whole-model-level checkpointing. Returns 1-D positive-class
    probability arrays: (oof_sent, oof_sarc, valid_sent, valid_sarc). OOF arrays are always
    aligned to the original train_df (length N_ORIGINAL_TRAIN), regardless of dataset_tag.'''
    import gc, tempfile, time
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

    n_folds = n_folds or N_FOLDS
    epochs = epochs or EPOCHS
    dataset_tag = dataset_tag or TRANSFORMER_DATASET_TAG
    ckpt_dir = CKPT_DIR_BY_TAG[dataset_tag]
    safe_name = _safe_ckpt_name(model_name)

    def _atomic_save(obj, path):
        directory = os.path.dirname(path)
        os.makedirs(directory, exist_ok=True)
        fd, tmp_path = tempfile.mkstemp(suffix=".tmp", dir=directory)
        os.close(fd)
        try:
            torch.save(obj, tmp_path)
            os.replace(tmp_path, path)
        finally:
            if os.path.exists(tmp_path):
                try:
                    os.remove(tmp_path)
                except Exception:
                    pass

    def _fold_dir():
        d = os.path.join(ckpt_dir, safe_name, f"folds_{n_folds}")
        os.makedirs(d, exist_ok=True)
        return d

    def _fold_ckpt_path(fold):
        return os.path.join(_fold_dir(), f"fold_{fold}.pt")

    def _fold_model_path(fold):
        return os.path.join(_fold_dir(), f"fold_{fold}_model.pt")

    def _load_fold_checkpoint(fold):
        path = _fold_ckpt_path(fold)
        if not os.path.exists(path):
            return None
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
            return payload if payload.get("complete") else None
        except Exception as e:
            print(f"  Could not load fold {fold} checkpoint: {e}")
            return None

    def _save_fold_checkpoint(fold, val_indices, val_sent_proba, val_sarc_proba):
        _atomic_save({
            "val_indices": np.asarray(val_indices),
            "val_sent_proba": np.asarray(val_sent_proba),
            "val_sarc_proba": np.asarray(val_sarc_proba),
            "complete": True, "timestamp": time.time(),
        }, _fold_ckpt_path(fold))
        print(f"  checkpoint saved: {_fold_ckpt_path(fold)}")

    def _complete_summary_path():
        return os.path.join(ckpt_dir, safe_name, f"complete_{n_folds}fold_outputs.npz")

    def _load_complete_summary():
        path = _complete_summary_path()
        if not os.path.exists(path):
            return None
        try:
            data = np.load(path)
            print(f"  Found complete cached result: {path}")
            return data["oof_sent"], data["oof_sarc"], data["valid_sent"], data["valid_sarc"]
        except Exception as e:
            print(f"  Could not load complete summary: {e}")
            return None

    def _save_complete_summary(oof_sent, oof_sarc, valid_sent, valid_sarc):
        path = _complete_summary_path()
        os.makedirs(os.path.dirname(path), exist_ok=True)
        np.savez_compressed(path, oof_sent=oof_sent, oof_sarc=oof_sarc, valid_sent=valid_sent, valid_sarc=valid_sarc)
        print(f"  Complete result cached: {path}")

    print(f"\n{'='*70}\n[{model_name}]  dataset={dataset_tag}  folds={n_folds}  epochs={epochs}\n{'='*70}")

    cached = _load_complete_summary()
    if cached is not None:
        print("  -> using cached complete result, no training needed.")
        return cached

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_fp16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}  |  FP16 autocast: {use_fp16}  |  model weights stay FP32")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class TextDataset(Dataset):
        def __init__(self, texts):
            self.texts = list(texts)
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            enc = tokenizer(str(self.texts[idx]), truncation=True, padding="max_length",
                             max_length=TRANSFORMER_MAX_LEN, return_tensors="pt")
            return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0),
                    "index": torch.tensor(idx, dtype=torch.long)}

    class MultiTaskEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(model_name)  # keep FP32 weights; mixed precision is applied via autocast only
            hidden = self.encoder.config.hidden_size
            self.dropout = nn.Dropout(0.1)
            self.sentiment_head = nn.Linear(hidden, 2)
            self.sarcasm_head = nn.Linear(hidden, 2)
        def forward(self, input_ids, attention_mask):
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
            pooled = self.dropout(pooled)
            return self.sentiment_head(pooled), self.sarcasm_head(pooled)

    pool_texts = TRANSFORMER_TRAIN_DF["text"].fillna("").astype(str).values
    pool_sentiment = TRANSFORMER_TRAIN_DF["sentiment"].astype(int).values
    pool_sarcasm = TRANSFORMER_TRAIN_DF["sarcasm"].astype(int).values
    parent_idx_array = TRANSFORMER_TRAIN_DF["_parent_idx"].values.astype(int)

    valid_texts = valid_df["text"].fillna("").astype(str).values
    valid_loader = DataLoader(TextDataset(valid_texts), batch_size=TRANSFORMER_BATCH_SIZE * 4, shuffle=False)

    oof_sent_proba = np.zeros(N_ORIGINAL_TRAIN, dtype=np.float32)
    oof_sarc_proba = np.zeros(N_ORIGINAL_TRAIN, dtype=np.float32)
    fold_valid_sent, fold_valid_sarc = [], []

    joint_label = train_df["sentiment"].values * 2 + train_df["sarcasm"].values
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)

    for fold, (orig_tr_idx, orig_va_idx) in enumerate(skf.split(np.arange(N_ORIGINAL_TRAIN), joint_label), start=1):
        print(f"\n=== [{model_name}/{dataset_tag}] Fold {fold}/{n_folds} ===")

        fold_ckpt = _load_fold_checkpoint(fold)
        model_path = _fold_model_path(fold)
        oof_ok = fold_ckpt is not None and np.array_equal(np.asarray(fold_ckpt["val_indices"]), orig_va_idx)
        if oof_ok:
            oof_sent_proba[orig_va_idx] = fold_ckpt["val_sent_proba"]
            oof_sarc_proba[orig_va_idx] = fold_ckpt["val_sarc_proba"]
        model_ok = os.path.exists(model_path)
        need_train = not (oof_ok and model_ok)

        if need_train:
            if fold_ckpt is not None and not oof_ok:
                print("  cached fold indices don't match the current split - retraining.")
            elif oof_ok and not model_ok:
                print("  OOF cached but fold model weights are missing - retraining so valid-set")
                print("  predictions can still be produced.")
            else:
                print("  no checkpoint found - training.")

            train_positions = np.where(np.isin(parent_idx_array, orig_tr_idx))[0]
            train_texts_fold = pool_texts[train_positions]
            train_sent_fold = pool_sentiment[train_positions]
            train_sarc_fold = pool_sarcasm[train_positions]
            heldout_texts_fold = pool_texts[orig_va_idx]

            train_loader = DataLoader(TextDataset(train_texts_fold), batch_size=TRANSFORMER_BATCH_SIZE,
                                       shuffle=True, num_workers=0, pin_memory=True)
            heldout_loader = DataLoader(TextDataset(heldout_texts_fold), batch_size=TRANSFORMER_BATCH_SIZE * 4,
                                         shuffle=False, num_workers=0, pin_memory=True)

            model = MultiTaskEncoder().to(device)
            if _is_deberta(model_name) and hasattr(model.encoder, "gradient_checkpointing_enable"):
                model.encoder.gradient_checkpointing_enable()
                if hasattr(model.encoder, "config"):
                    model.encoder.config.use_cache = False
                print("  gradient checkpointing enabled (deberta family)")

            optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LR, weight_decay=TRANSFORMER_WEIGHT_DECAY)
            steps_per_epoch = max(1, int(np.ceil(len(train_loader) / TRANSFORMER_GRAD_ACCUMULATION)))
            total_steps = steps_per_epoch * epochs
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * TRANSFORMER_WARMUP_RATIO),
                                                         num_training_steps=total_steps)
            sent_loss_fn = nn.CrossEntropyLoss()
            sarc_loss_fn = nn.CrossEntropyLoss()
            scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)

            model.train()
            for epoch in range(epochs):
                epoch_loss, n_skipped = 0.0, 0
                optimizer.zero_grad(set_to_none=True)
                for step, batch in enumerate(train_loader):
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attn = batch["attention_mask"].to(device, non_blocking=True)
                    local_idx = batch["index"].numpy()
                    sentiment_target = torch.tensor(train_sent_fold[local_idx], dtype=torch.long, device=device)
                    sarcasm_target = torch.tensor(train_sarc_fold[local_idx], dtype=torch.long, device=device)

                    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_fp16):
                        sent_logits, sarc_logits = model(input_ids, attn)
                        loss = (sent_loss_fn(sent_logits, sentiment_target) +
                                sarc_loss_fn(sarc_logits, sarcasm_target)) / TRANSFORMER_GRAD_ACCUMULATION

                    if not torch.isfinite(loss):
                        # Skip batches with NaN/Inf loss (a known risk for DeBERTa-family models)
                        # rather than propagating a corrupted gradient.
                        n_skipped += 1
                        optimizer.zero_grad(set_to_none=True)
                        continue

                    scaler.scale(loss).backward()
                    is_accum_step = (step + 1) % TRANSFORMER_GRAD_ACCUMULATION == 0
                    is_last_step = (step + 1) == len(train_loader)
                    if is_accum_step or is_last_step:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), TRANSFORMER_MAX_GRAD_NORM)
                        scaler.step(optimizer)
                        scaler.update()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)
                    epoch_loss += loss.item() * TRANSFORMER_GRAD_ACCUMULATION

                if n_skipped:
                    print(f"    (skipped {n_skipped} numerically unstable batch(es))")
                print(f"  epoch {epoch+1}/{epochs}  loss={epoch_loss / max(1, len(train_loader)):.4f}")

            model.eval()
            sent_probs, sarc_probs = [], []
            with torch.no_grad():
                for batch in heldout_loader:
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attn = batch["attention_mask"].to(device, non_blocking=True)
                    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_fp16):
                        sl, sal = model(input_ids, attn)
                    sent_probs.append(torch.softmax(sl, dim=-1)[:, 1].float().cpu().numpy())
                    sarc_probs.append(torch.softmax(sal, dim=-1)[:, 1].float().cpu().numpy())
            fold_sent_proba = np.concatenate(sent_probs)
            fold_sarc_proba = np.concatenate(sarc_probs)
            oof_sent_proba[orig_va_idx] = fold_sent_proba
            oof_sarc_proba[orig_va_idx] = fold_sarc_proba

            _save_fold_checkpoint(fold, orig_va_idx, fold_sent_proba, fold_sarc_proba)
            _atomic_save({"model_state_dict": model.state_dict()}, model_path)
            print(f"  fold {fold} model saved.")
        else:
            print("  found completed fold (OOF + model) - skipping training.")
            model = MultiTaskEncoder()
            state = torch.load(model_path, map_location="cpu", weights_only=True)
            model.load_state_dict(state["model_state_dict"])
            model.to(device)

        model.eval()
        vsent, vsarc = [], []
        with torch.no_grad():
            for batch in valid_loader:
                input_ids = batch["input_ids"].to(device, non_blocking=True)
                attn = batch["attention_mask"].to(device, non_blocking=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_fp16):
                    sl, sal = model(input_ids, attn)
                vsent.append(torch.softmax(sl, dim=-1)[:, 1].float().cpu().numpy())
                vsarc.append(torch.softmax(sal, dim=-1)[:, 1].float().cpu().numpy())
        fold_valid_sent.append(np.concatenate(vsent))
        fold_valid_sarc.append(np.concatenate(vsarc))

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  fold {fold}/{n_folds} complete.")

    valid_sent_proba = np.mean(fold_valid_sent, axis=0)
    valid_sarc_proba = np.mean(fold_valid_sarc, axis=0)
    _save_complete_summary(oof_sent_proba, oof_sarc_proba, valid_sent_proba, valid_sarc_proba)
    print(f"\n[{model_name}/{dataset_tag}] complete.")
    return oof_sent_proba, oof_sarc_proba, valid_sent_proba, valid_sarc_proba


if HAS_TORCH and HAS_TRANSFORMERS and HAS_CUDA:
    (transformer_oof_sent_proba, transformer_oof_sarc_proba,
     transformer_valid_sent_proba, transformer_valid_sarc_proba) = run_transformer_cv_pipeline(TRANSFORMER_MODEL_NAME)
else:
    print("Skipping Transformer training: a GPU and Hugging Face Hub access are required.")
    print("\nTo run this section:")
    print("  1. Use a GPU runtime (e.g. Google Colab: Runtime > Change runtime type > GPU).")
    print("  2. pip install torch transformers")
    print(f"  3. Re-run this cell to train or resume {TRANSFORMER_MODEL_NAME} on the")
    print(f"     '{TRANSFORMER_DATASET_TAG}' training set. Existing checkpoints are reused automatically.")
    transformer_oof_sent_proba = transformer_oof_sarc_proba = None
    transformer_valid_sent_proba = transformer_valid_sarc_proba = None

Skipping Transformer training: a GPU and Hugging Face Hub access are required.

To run this section:
  1. Use a GPU runtime (e.g. Google Colab: Runtime > Change runtime type > GPU).
  2. pip install torch transformers
  3. Re-run this cell to train or resume roberta-base on the
     'classical_aug' training set. Existing checkpoints are reused automatically.


## 9. Per-dialect threshold tuning and final predictions from the Transformer

The same principle as Section 4 applied to the Transformer: one threshold per dialect per task is tuned on the **out-of-fold** probabilities from Section 8 (never on `valid.csv` labels), then applied to the fold-averaged predictions on `valid.csv`.
The Transformer is also blended with the classical model's probabilities, since averaging differently-biased models typically improves macro-F1.

In [16]:
def tune_threshold_per_dialect(task, oof_proba, oof_df):
    y = oof_df[task].values
    variety = oof_df["variety"].values
    thresholds = {}
    for dialect in ["en-AU", "en-UK"]:
        mask = variety == dialect
        best_thr, best_f1 = 0.5, -1.0
        for thr in np.arange(0.10, 0.91, 0.02):
            pred = (oof_proba[mask] >= thr).astype(int)
            f1 = f1_score(y[mask], pred, average="macro")
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[dialect] = best_thr
        print(f"[{task}] {dialect}: best OOF threshold = {best_thr:.2f} (OOF macro-F1 = {best_f1:.4f})")
    return thresholds

if transformer_valid_sent_proba is not None:
    tf_thr_sent = tune_threshold_per_dialect("sentiment", transformer_oof_sent_proba, train_df)
    tf_thr_sarc = tune_threshold_per_dialect("sarcasm", transformer_oof_sarc_proba, train_df)

    transformer_pred = make_answer_df(
        valid_df,
        apply_per_dialect_threshold("sentiment", transformer_valid_sent_proba, tf_thr_sent),
        apply_per_dialect_threshold("sarcasm", transformer_valid_sarc_proba, tf_thr_sarc),
    )
    print("\nTransformer (5-fold ensemble, per-dialect tuned thresholds):")
    _, transformer_final = score_predictions(transformer_pred, valid_df)

    # Blend with the classical model's probabilities (Sections 3-5)
    ensemble_sent_proba = 0.5 * transformer_valid_sent_proba + 0.5 * valid_proba["sentiment"]
    ensemble_sarc_proba = 0.5 * transformer_valid_sarc_proba + 0.5 * valid_proba["sarcasm"]
    ensemble_pred = make_answer_df(
        valid_df,
        apply_per_dialect_threshold("sentiment", ensemble_sent_proba, tf_thr_sent),
        apply_per_dialect_threshold("sarcasm", ensemble_sarc_proba, tf_thr_sarc),
    )
    print("\nTransformer + classical ensemble:")
    _, ensemble_final = score_predictions(ensemble_pred, valid_df)

    final_pred = ensemble_pred if ensemble_final >= transformer_final else transformer_pred
    save_submission(final_pred, "dev_transformer_best")
else:
    print("No Transformer probabilities available in this run; see Section 8.")
    print(f"Current best submission is the classical baseline: {zip_path}")

No Transformer probabilities available in this run; see Section 8.
Current best submission is the classical baseline: ../submissions/dev_classical_baseline.zip


## Extensions

The following sections cover additional techniques for improving robustness and accuracy.

> **Validation note.** `valid.csv` is useful for calibration, but a model iterated on heavily against it may generalise worse to an unseen test set. Treat the cross-validation (OOF) scores produced in this notebook, which never use `valid.csv` labels for tuning, as the primary signal, and the validation-set score as a secondary check.

## 10. Pseudo-labelling on unlabelled in-domain text

If unlabelled in-domain text is available (for example, the blind test input once released, or additional text from the same domains), predicting on it with the best model and retaining only high-confidence predictions can expand the training set.
Pseudo-label quality should be validated carefully, as noisy labels can hurt the minority class. The function below is provided as a utility; it requires unlabelled data that is not included with this task.

In [17]:
def pseudo_label_and_augment(sent_proba, sarc_proba, source_df, confidence_threshold=0.9):
    '''Keeps only high-confidence predictions on UNLABELLED text and returns them as new
    pseudo-labeled rows in the source/variety/text/sentiment/sarcasm shape. Only ever call
    this on text you don't already have real labels for - valid.csv does not qualify
    because it is labelled.'''
    sent_conf = np.maximum(sent_proba, 1 - sent_proba)
    sarc_conf = np.maximum(sarc_proba, 1 - sarc_proba)
    keep = (sent_conf >= confidence_threshold) & (sarc_conf >= confidence_threshold)
    pseudo = source_df.loc[keep, ["source", "variety", "text"]].copy()
    pseudo["sentiment"] = (np.asarray(sent_proba)[keep] >= 0.5).astype(int)
    pseudo["sarcasm"] = (np.asarray(sarc_proba)[keep] >= 0.5).astype(int)
    print(f"Pseudo-labeled {keep.sum()}/{len(source_df)} rows at confidence >= {confidence_threshold} "
          f"({100*keep.mean():.1f}% kept)")
    return pseudo

# Example usage, given unlabelled text and Transformer probabilities for it:
#   pseudo_df = pseudo_label_and_augment(sent_proba, sarc_proba, unlabelled_df, confidence_threshold=0.9)
#   train_df_expanded = pd.concat([train_df, pseudo_df], ignore_index=True)  # then retrain (Section 8)

## 11. Multi-encoder ensemble

Averaging probabilities from different encoder architectures generally helps more than averaging seeds of a single architecture. Every encoder is trained through `run_transformer_cv_pipeline()` from Section 8, so the same leakage-safe folds, checkpointing and NaN/Inf guards apply
(the guards matter most for `deberta-v3-large`, which is prone to numerical instability).

In [18]:
ENCODER_ENSEMBLE = [
    # (model name, number of folds, epochs)
    ("roberta-base", 5, 5),
    ("google/electra-base-discriminator", 5, 5),
    ("cardiffnlp/twitter-roberta-base-sentiment-latest", 5, 4),
    # ("microsoft/deberta-v3-large", 3, 3),  # optional: slower and more prone to numerical instability
]

if HAS_TORCH and HAS_TRANSFORMERS and HAS_CUDA:
    encoder_oof_sent, encoder_oof_sarc = [], []
    encoder_valid_sent, encoder_valid_sarc = [], []
    for name, n_folds, epochs in ENCODER_ENSEMBLE:
        print(f"\n{'='*60}\nEncoder: {name}\n{'='*60}")
        oof_s, oof_c, val_s, val_c = run_transformer_cv_pipeline(
            model_name=name, n_folds=n_folds, epochs=epochs, dataset_tag=TRANSFORMER_DATASET_TAG)
        encoder_oof_sent.append(oof_s); encoder_oof_sarc.append(oof_c)
        encoder_valid_sent.append(val_s); encoder_valid_sarc.append(val_c)

    multi_oof_sent = np.mean(encoder_oof_sent, axis=0)
    multi_oof_sarc = np.mean(encoder_oof_sarc, axis=0)
    multi_valid_sent = np.mean(encoder_valid_sent, axis=0)
    multi_valid_sarc = np.mean(encoder_valid_sarc, axis=0)

    multi_thr_sent = tune_threshold_per_dialect("sentiment", multi_oof_sent, train_df)
    multi_thr_sarc = tune_threshold_per_dialect("sarcasm", multi_oof_sarc, train_df)
    multi_pred = make_answer_df(
        valid_df,
        apply_per_dialect_threshold("sentiment", multi_valid_sent, multi_thr_sent),
        apply_per_dialect_threshold("sarcasm", multi_valid_sarc, multi_thr_sarc),
    )
    print("\nMulti-encoder ensemble:")
    _, multi_final = score_predictions(multi_pred, valid_df)
    save_submission(multi_pred, "dev_multi_encoder_ensemble")
else:
    print("Skipping multi-encoder ensembling: a GPU and Hugging Face Hub access are required.")
    print("Each entry in ENCODER_ENSEMBLE is trained with run_transformer_cv_pipeline() (Section 8);")
    print("completed encoders are loaded from checkpoints rather than retrained.")

Skipping multi-encoder ensembling: a GPU and Hugging Face Hub access are required.
Each entry in ENCODER_ENSEMBLE is trained with run_transformer_cv_pipeline() (Section 8);
completed encoders are loaded from checkpoints rather than retrained.


## 12. Error analysis

This section needs no GPU. It inspects the classical model's false negatives and false positives on `valid.csv` for en-UK sarcasm, the hardest slice, to identify systematic error patterns.

In [19]:
sarc_pred_final = (valid_proba["sarcasm"] >= 0.5).astype(int)
uk_mask = (valid_df["variety"] == "en-UK").values
fn_mask = uk_mask & (valid_df["sarcasm"].values == 1) & (sarc_pred_final == 0)
fp_mask = uk_mask & (valid_df["sarcasm"].values == 0) & (sarc_pred_final == 1)

print(f"en-UK sarcasm: {fn_mask.sum()} false negatives (missed sarcasm), {fp_mask.sum()} false positives (over-flagged)")

print("\nFalse negatives - actually sarcastic, model said 'no':")
for t in valid_df.loc[fn_mask, "text"].head(6):
    print(" -", str(t)[:160].replace("\n", " "))

print("\nFalse positives - model said 'sarcastic', actually not:")
for t in valid_df.loc[fp_mask, "text"].head(6):
    print(" -", str(t)[:160].replace("\n", " "))


en-UK sarcasm: 9 false negatives (missed sarcasm), 49 false positives (over-flagged)

False negatives - actually sarcastic, model said 'no':
 - Even if it does. They go in as non combatant officers. Or almost directly fast tracked to admiral and crash a 4billion quid ship  sorry sorry "mechanical fault"
 - This is **absolutely** on the past 14 years of tory policies. There ***is*** central funding to support the hiring on physician associates and other "not doctor
 - Welcome to western values. The scales are sliding and based solely upon who pays more to whom. If you're the designated bad guy, then you are responsible for ev
 - I get the Independent but have recently decided to change to a different paper; it's not very absorbent you see.  Oh, I forgot to mention, I only buy it for my 
 - A culture of service is a good thing for our country' as he sneaks out the door. What a shit.
 - Euro Garages will have training in advanced shelf stacking, on top of their existing Customer Services 

**Observed error patterns.** False positives tend to involve strong or profane language used *sincerely* (genuine enthusiasm or complaint), which the model has learned to associate with sarcasm.
False negatives tend to be dry, deadpan political or social commentary with no exclamation marks or other surface markers. The first pattern can be reduced through class weighting and threshold tuning;
the second requires contextual understanding, which is what the Transformer in Section 8 provides. A shallow lexical model cannot reliably detect deadpan sarcasm regardless of the decision threshold.

## 13. Submission format

- Submissions are a zip file containing a single `answer.csv`, with the same columns and row order as the input file (`source`, `variety`, `text`, `sentiment`, `sarcasm`).
- `save_submission()` (Section 2) writes both `<name>.csv` and `<name>.zip` to the `submissions/` folder.
- Predictions must be produced automatically from the input text; no test labels may be used.